# 01 — Korpus & Annotation

Phase 1 (Korpus + Bias-Notiz) und Phase 2 (κ + drei Edge Cases) leben in diesem Notebook. Pipeline → `02_extract.ipynb`, Eval → `03_eval.ipynb`, Frontier → `04_frontier_compare.ipynb`.

## Run-Header

| Feld | Wert |
|---|---|
| Datum (Phase 1) | 2026-05-11 |
| Datum (Phase 2) | _YYYY-MM-DD_ |
| Korpus-Datei | `daten/eigener_korpus.jsonl` |
| Anzahl Anzeigen im Korpus | 44 |
| Genutzte Suchanfragen (Phase 1) | _ |
| Pair-Partner:in (Phase 2) | _ |
| 12 gemeinsame Anzeigen-IDs | _ |

## Phase 1 — Korpus inspizieren + Bias-Notiz

In [4]:
import requests
import json
import time
import base64
from pathlib import Path

API_KEY = "jobboerse-jobsuche"
BASE_URL = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4"

HEADERS = {
    "X-API-Key": API_KEY
}

suchanfragen = [
    {"was": "Data Analyst", "wo": "Bremen"},
    {"was": "Data Engineer", "wo": "Bremen"},
    {"was": "Business Intelligence", "wo": "Bremen"},
    {"was": "Fachinformatiker Daten- und Prozessanalyse", "wo": "Bremen"},
    {"was": "Data Scientist", "wo": "Bremen"},
]

anzeigen = {}

for suche in suchanfragen:
    print("Suche:", suche)

    response = requests.get(
        f"{BASE_URL}/jobs",
        headers=HEADERS,
        params={
            "was": suche["was"],
            "wo": suche["wo"],
            "size": 20,
        }
    )

    print("Status:", response.status_code)

    daten = response.json() # dictonary aus Antwort erstellen
    treffer = daten.get("stellenangebote", []) # Liste mit Stellenangeboten ziehen
    
# Detailsuche für gefundene Treffer anhand Refnr um an stellenangebotsBeschreibung zu kommen
    for treffer_item in treffer:
        refnr = treffer_item.get("refnr")

        if not refnr:
            continue

        if refnr in anzeigen: # falls Anzeige bereits mit anderem Suchbegriff gefunden wurde und doppelt auftaucht
            continue

        refnr_encoded = base64.b64encode(refnr.encode("utf-8")).decode("utf-8") # damit das Anhängen an die URL keine Fehler durch Sonderzeichen wirft

        detail_response = requests.get(
            f"{BASE_URL}/jobdetails/{refnr_encoded}",
            headers=HEADERS
        )

        if detail_response.status_code != 200:
            print("Detail fehlgeschlagen:", refnr, detail_response.status_code)
            continue

        detail = detail_response.json()

        anzeige = {
            "refnr": refnr,
            "titel": detail.get("stellenangebotsTitel"),
            "firma": detail.get("firma"),
            "text": detail.get("stellenangebotsBeschreibung"),
            "ort": detail.get("stellenlokationen", [{}])[0].get("adresse", {}).get("ort"),
            "beruf": detail.get("hauptberuf"),
            "veroeffentlichung": detail.get("datumErsteVeroeffentlichung"),
            "homeoffice_api": detail.get("homeofficemoeglich"),
            "gehalt_api": detail.get("verguetungsangabe"),
            "vertragsdauer_api": detail.get("vertragsdauer"),
            "raw": detail # Originalantwort der API
        }

        if anzeige["text"]:
            anzeigen[refnr] = anzeige

        time.sleep(0.5) # Pause zwischen Abfragen

print("Gesammelte Anzeigen:", len(anzeigen))

Suche: {'was': 'Data Analyst', 'wo': 'Bremen'}
Status: 200
Suche: {'was': 'Data Engineer', 'wo': 'Bremen'}
Status: 200
Suche: {'was': 'Business Intelligence', 'wo': 'Bremen'}
Status: 200
Suche: {'was': 'Fachinformatiker Daten- und Prozessanalyse', 'wo': 'Bremen'}
Status: 200
Suche: {'was': 'Data Scientist', 'wo': 'Bremen'}
Status: 200
Gesammelte Anzeigen: 44


In [5]:
output_path = Path("../daten/eigener_korpus.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f: #w -> überschreiben
    for anzeige in anzeigen.values():
        f.write(json.dumps(anzeige, ensure_ascii=False) + "\n") # ensure_ascii=False -> Umlaute erlauben; + "\n" danach neue Zeile

print("Gespeichert:", output_path)
print("Anzahl gespeicherter Anzeigen:", len(anzeigen))

Gespeichert: ../daten/eigener_korpus.jsonl
Anzahl gespeicherter Anzeigen: 44


In [6]:
with open("../daten/eigener_korpus.jsonl", "r", encoding="utf-8") as f:
    zeilen = f.readlines()

print("Zeilen in Datei:", len(zeilen))

erste_anzeige = json.loads(zeilen[0])
erste_anzeige.keys()

Zeilen in Datei: 44


dict_keys(['refnr', 'titel', 'firma', 'text', 'ort', 'beruf', 'veroeffentlichung', 'homeoffice_api', 'gehalt_api', 'vertragsdauer_api', 'raw'])

## Phase 2 — κ-Tabelle + drei Edge Cases